# Digifly Phase 2 Workbench

This is the notebook-first public control surface for Phase 2.

It is built around a simple rule: keep the simulation logic in Python modules, and keep the notebook thin. That gives you something that works like an app inside Jupyter right now, without trapping the project inside one giant notebook.

Use this notebook when you want to:

- launch shared Phase 2 runs (`single`, `custom`, `hemilineage`)
- launch the fuller hemilineage project pipeline
- preview and validate run plans before execution
- save workbench bundles and manifests for reproducibility
- keep error and fix logs next to the run

Recommended workflow:

1. Point the workbench at your Phase 1 `export_swc` tree.
2. Pick the closest preset.
3. Preview the plan.
4. Validate.
5. Write the bundle.
6. Run.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def _safe_cwd() -> Path:
    try:
        return Path(os.getcwd())
    except FileNotFoundError:
        home = os.environ.get('HOME')
        if home:
            return Path(home)
        return Path('/tmp')


def _find_phase2_root() -> Path:
    candidates = []

    env_root = os.environ.get('DIGIFLY_PHASE2_ROOT', '').strip()
    if env_root:
        candidates.append(Path(env_root))

    cwd = _safe_cwd().expanduser().resolve()
    candidates.append(cwd)
    candidates.extend(cwd.parents)

    notebook_hint = Path('/Users/juanlopez2016/Desktop/Digifly Public/Phase 2').resolve()
    candidates.append(notebook_hint)

    seen = set()
    for candidate in candidates:
        try:
            candidate = candidate.expanduser().resolve()
        except Exception:
            continue
        if str(candidate) in seen:
            continue
        seen.add(str(candidate))
        if (candidate / 'digifly').exists():
            return candidate
    raise RuntimeError('Could not find the Phase 2 root. Set DIGIFLY_PHASE2_ROOT to the folder that contains digifly/.')


PHASE2_ROOT = _find_phase2_root()
if str(PHASE2_ROOT) not in sys.path:
    sys.path.insert(0, str(PHASE2_ROOT))

print(f'Phase 2 root: {PHASE2_ROOT}')

from digifly.phase2.workbench import PRESETS, launch_workbench

print('Available presets:')
for preset in PRESETS:
    print(f'  - {preset.slug}: {preset.label}')


## Workbench Shape

The workbench currently gives you two execution surfaces:

- `shared_runner`: the current `run_walking_simulation()` path for `single`, `custom`, and `hemilineage`
- `hemilineage_project`: the fuller `run_full_hemilineage_project()` path for baseline and pipeline-style Phase 2 work

The quick controls cover the common knobs. The advanced JSON boxes are there so you can still reach deeper features without turning the notebook into a wall of code.

Artifacts are written in two places:

- a workbench preflight bundle under `Phase 2/workbench_runs/<run_id>/`
- the actual simulation output folder written by the selected runner

Both locations get manifest-style files, and the workbench creates `error_log.md` and `fix_log.md` placeholders so each run has a paper trail.


In [ ]:
ui = launch_workbench(PHASE2_ROOT)
ui


## When This Should Become An App

You do not need to jump to a standalone app yet.

Best practice here is:

1. keep building the Phase 2 control schema in Python modules
2. keep testing it through this notebook workbench
3. once the presets and manifests feel stable, wrap the same widget layer with a notebook app surface such as Voilà
4. only build a separate web app if the project truly needs multi-page navigation, long-lived background jobs, or collaboration features

That way the notebook is not a dead-end. It is version 1 of the app surface.


In [ ]:
# Optional helper: inspect the latest workbench bundle path.
from pathlib import Path

workbench_runs = (PHASE2_ROOT / 'workbench_runs').resolve()
workbench_runs.mkdir(parents=True, exist_ok=True)
latest = sorted(workbench_runs.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)
latest[0] if latest else 'No workbench bundles yet.'
